In [0]:
import pandas as pd
raw_path = '/Volumes/airfly_workspace/default/airfly_insights/airfly _raw_data.csv'
df = pd.read_csv(raw_path)
print("Raw shape:", df.shape)


Raw shape: (484551, 29)


In [0]:
# Handle Nulls in Delay and Cancellation Columns

import numpy as np

# Fill delay columns with 0
delay_cols = ['ArrDelay', 'DepDelay', 'CarrierDelay', 'WeatherDelay', 
              'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
df[delay_cols] = df[delay_cols].fillna(0)

# Standardize cancellation columns
df['Cancelled'] = df['Cancelled'].map({'Y': 1, 'N': 0})
df['Diverted'] = df['Diverted'].fillna(0)
df['CancellationCode'] = df['CancellationCode'].fillna('None')


In [0]:
# Format Date and Time Columns

# Use dayfirst=True since format is DD-MM-YYYY
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Convert DepTime, ArrTime, CRSArrTime from HHMM integer to time
def convert_hhmm(time):
    try:
        time = int(time)
        return pd.to_datetime(f'{time:04}', format='%H%M').time()
    except:
        return pd.NaT

for col in ['DepTime', 'ArrTime', 'CRSArrTime']:
    df[col] = df[col].apply(convert_hhmm)

# Quick check
print(df[['Date', 'DepTime', 'ArrTime', 'CRSArrTime']].head())


        Date   DepTime   ArrTime CRSArrTime
0 2019-01-03  18:29:00  19:59:00   19:25:00
1 2019-01-03  19:37:00  20:37:00   19:40:00
2 2019-01-03  16:44:00  18:45:00   17:25:00
3 2019-01-03  14:52:00  16:40:00   16:25:00
4 2019-01-03  13:23:00  15:26:00   15:10:00


In [0]:
# Create Derived Features

# Ensure Date is datetime
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')  # invalid parsing → NaT

# Drop rows where Date could not be parsed 
df = df.dropna(subset=['Date'])

# Now create derived features
df['Month'] = df['Date'].dt.month
df['DayOfWeekNum'] = df['Date'].dt.dayofweek  # Monday=0
df['DepHour'] = df['DepTime'].apply(lambda x: x.hour if pd.notnull(x) else np.nan)
df['Route'] = df['Origin'] + '-' + df['Dest']

# Quick check
print(df[['Date','Month','DayOfWeekNum','DepHour','Route']].head())

        Date  Month  DayOfWeekNum  DepHour    Route
0 2019-01-03      1             3     18.0  IND-BWI
1 2019-01-03      1             3     19.0  IND-LAS
2 2019-01-03      1             3     16.0  IND-MCO
3 2019-01-03      1             3     14.0  IND-PHX
4 2019-01-03      1             3     13.0  IND-TPA


In [0]:
# Remove Duplicates

# Drop duplicate flights (same Date, FlightNum, TailNum, Origin, Dest)
df.drop_duplicates(subset=['Date', 'FlightNum', 'TailNum', 'Origin', 'Dest'], inplace=True)

In [0]:
# Ensure Numeric Columns Are Correct

numeric_cols = ['ActualElapsedTime','CRSElapsedTime','AirTime','TaxiIn','TaxiOut'] + delay_cols
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [0]:
# Quick Exploration Examples

# Average arrival delay by airline
print(df.groupby('Airline')['ArrDelay'].mean().sort_values(ascending=False))

# Cancellation rate by month
print(df.groupby('Month')['Cancelled'].mean())

# Most popular routes
print(df['Route'].value_counts().head(10))

Airline
JetBlue Airways                 72.869370
United Air Lines Inc.           69.670539
American Airlines Inc.          65.730941
Skywest Airlines Inc.           65.187659
American Eagle Airlines Inc.    64.277233
Atlantic Southeast Airlines     63.210684
Delta Air Lines Inc.            59.292607
US Airways Inc.                 58.454165
Alaska Airlines Inc.            57.557600
Hawaiian Airlines Inc.          55.658333
Southwest Airlines Co.          51.032945
Frontier Airlines Inc.          41.973150
Name: ArrDelay, dtype: float64
Month
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
Name: Cancelled, dtype: float64
ORD-LGA    1920
LGA-ORD    1615
LAX-SFO    1603
SFO-LAX    1457
LAS-LAX    1305
HOU-DAL    1276
DAL-HOU    1200
ORD-LAX    1154
PHX-LAS    1152
DFW-ORD    1125
Name: Route, dtype: int64


In [0]:
# Check for any missing values

# Check total missing values per column
print(df.isna().sum())

DayOfWeek                 0
Date                      0
DepTime                  90
ArrTime                 464
CRSArrTime                0
UniqueCarrier             0
Airline                   0
FlightNum                 0
TailNum                   0
ActualElapsedTime         0
CRSElapsedTime            0
AirTime                   0
ArrDelay                  0
DepDelay                  0
Origin                    0
Org_Airport            1177
Dest                      0
Dest_Airport           1479
Distance                  0
TaxiIn                    0
TaxiOut                   0
Cancelled            484545
CancellationCode          0
Diverted                  0
CarrierDelay              0
WeatherDelay              0
NASDelay                  0
SecurityDelay             0
LateAircraftDelay         0
Month                     0
DayOfWeekNum              0
DepHour                  90
Route                     0
dtype: int64


In [0]:
# Handle Missing DepTime / ArrTime / DepHour

# Fill missing times with NaT (already converted), or optionally drop these rows if very few
df = df.dropna(subset=['DepTime', 'ArrTime'])

# Update DepHour again after dropping rows
df['DepHour'] = df['DepTime'].apply(lambda x: x.hour if pd.notnull(x) else np.nan)

In [0]:
# Handle Missing Airport Names

df['Org_Airport'] = df['Org_Airport'].fillna('Unknown')
df['Dest_Airport'] = df['Dest_Airport'].fillna('Unknown')


In [0]:
# Handle Missing Cancelled

# Fill missing Cancelled values with 0 (not cancelled)
df['Cancelled'] = df['Cancelled'].fillna(0)


In [0]:
# Verify again

# Check total missing values
print(df.isna().sum())

# Quick overall check
print("Any missing values left?", df.isna().any().any())

DayOfWeek            0
Date                 0
DepTime              0
ArrTime              0
CRSArrTime           0
UniqueCarrier        0
Airline              0
FlightNum            0
TailNum              0
ActualElapsedTime    0
CRSElapsedTime       0
AirTime              0
ArrDelay             0
DepDelay             0
Origin               0
Org_Airport          0
Dest                 0
Dest_Airport         0
Distance             0
TaxiIn               0
TaxiOut              0
Cancelled            0
CancellationCode     0
Diverted             0
CarrierDelay         0
WeatherDelay         0
NASDelay             0
SecurityDelay        0
LateAircraftDelay    0
Month                0
DayOfWeekNum         0
DepHour              0
Route                0
dtype: int64
Any missing values left? False


In [0]:
# Save Preprocessed Dataset

# Save as Parquet for fast reuse
df.to_parquet('flights_cleaned.parquet', index=False)

# save as CSV
df.to_csv('flights_cleaned.csv', index=False)

In [0]:
# Save cleaned file to DBFS FileStore for download

cleaned_path = '/Volumes/airfly_workspace/default/airfly_insights/flights_cleaned.csv'
df.to_csv(cleaned_path, index=False)
print(f"Cleaned file saved to: {cleaned_path}")


Cleaned file saved to: /Volumes/airfly_workspace/default/airfly_insights/flights_cleaned.csv
